# Normalizzazione Fumetti DC e Marvel

In questa sezione procediamo all'unificazione e normalizzazione dei dataset dei fumetti provenienti da **DC Comics** (`DC_Comics_clean.csv`) e **Marvel Comics** (`Marvel_Comics_clean.csv`).

L'obiettivo è ottenere una struttura dati omogenea, priva di ambiguità e pronta per l'importazione nei nodi `:Issue` e `:Series` del database a grafi **Neo4j**.



In [20]:
from pathlib import Path
import pandas as pd

# Definizione dei percorsi
BASE_PATH = Path('/Users/marcogreco/ComicGraph')
PULITI_PATH = BASE_PATH / 'Dataset' / 'Puliti'

# Caricamento dei dataset puliti dei fumetti
dc_comics = pd.read_csv(PULITI_PATH / 'DC_Comics_clean.csv')
marvel_comics = pd.read_csv(PULITI_PATH / 'Marvel_Comics_clean.csv')

# Stampa di verifica
print(f"DC Comics caricati: {len(dc_comics)} righe, {len(dc_comics.columns)} colonne")
print(f"Marvel Comics caricati: {len(marvel_comics)} righe, {len(marvel_comics.columns)} colonne")
print("\nColonne DC:", dc_comics.columns.tolist())
print("Colonne Marvel:", marvel_comics.columns.tolist())

DC Comics caricati: 52812 righe, 14 colonne
Marvel Comics caricati: 34853 righe, 9 colonne

Colonne DC: ['Catergory_Title', 'Issue_Name', 'Issue_Link', 'Pencilers', 'Cover_Artists', 'Inkers', 'Writers', 'Editors', 'Executive_Editor', 'Letterers', 'Colourists', 'Release_Date', 'Comic_Series', 'Comic_Type']
Colonne Marvel: ['comic_name', 'active_years', 'issue_title', 'publish_date', 'issue_description', 'penciler', 'writer', 'cover_artist', 'Format']


In [21]:
# Aggiunta della colonna publisher per distinguere i due editori (come per i personaggi)
dc_comics['publisher'] = 'DC'
marvel_comics['publisher'] = 'Marvel'

# Creazione di una chiave primaria univoca globale per ogni albo
# indice di riga prefissato con l'editore
dc_comics['issue_id'] = 'dc_issue_' + dc_comics.reset_index().index.astype(str)
marvel_comics['issue_id'] = 'marvel_issue_' + marvel_comics.reset_index().index.astype(str)


dc_ids = set(dc_comics['issue_id'])
marvel_ids = set(marvel_comics['issue_id'])
print(f"ID DC creati: {len(dc_ids)}")
print(f"ID Marvel creati: {len(marvel_ids)}")
print(f"Sovrapposizioni: {len(dc_ids.intersection(marvel_ids))} (atteso: 0)")

ID DC creati: 52812
ID Marvel creati: 34853
Sovrapposizioni: 0 (atteso: 0)


In [22]:
# Eliminazione delle colonne secondarie da DC colonne non presenti in Marvel, sono ruoli secondari non indispensabili
dc_comics = dc_comics.drop(columns=[
    'Inkers',
    'Colourists',
    'Letterers',
    'Editors',
    'Executive_Editor',
    'Comic_Series'      
])



print("Colonne DC rimaste:", dc_comics.columns.tolist())



Colonne DC rimaste: ['Catergory_Title', 'Issue_Name', 'Issue_Link', 'Pencilers', 'Cover_Artists', 'Writers', 'Release_Date', 'Comic_Type', 'publisher', 'issue_id']


In [23]:
# Eliminazione delle colonne esclusive di Marvel non presenti in DC
marvel_comics = marvel_comics.drop(columns=[
    'active_years',       # parzialmente ridondante con comic_name
    'issue_description'   # testo libero non presente in DC
])



print("Colonne Marvel rimaste:", marvel_comics.columns.tolist())

Colonne Marvel rimaste: ['comic_name', 'issue_title', 'publish_date', 'penciler', 'writer', 'cover_artist', 'Format', 'publisher', 'issue_id']


In [24]:
# Eliminazione di Issue_Link da DC (non presente in Marvel)
dc_comics = dc_comics.drop(columns=['Issue_Link'])

# Ridenominazione colonne DC Comics in snake_case
dc_comics = dc_comics.rename(columns={
    'Catergory_Title': 'series_title',
    'Issue_Name': 'issue_title',
    'Release_Date': 'release_date',
    'Comic_Type': 'comic_type',
    'Writers': 'writer',
    'Pencilers': 'penciler',
    'Cover_Artists': 'cover_artist'
})

# Ridenominazione colonne Marvel Comics in snake_case
marvel_comics = marvel_comics.rename(columns={
    'comic_name': 'series_title',
    'publish_date': 'release_date',
    'Format': 'comic_type'
})

# Ordinamento identico delle colonne per entrambi i dataset (9 colonne omologate)
COLUMNS_ORDER = [
    'issue_id',
    'publisher',
    'series_title',
    'issue_title',
    'release_date',
    'comic_type',
    'writer',
    'penciler',
    'cover_artist'
]

dc_comics = dc_comics[COLUMNS_ORDER]
marvel_comics = marvel_comics[COLUMNS_ORDER]

print("Colonne DC:    ", dc_comics.columns.tolist())
print("Colonne Marvel:", marvel_comics.columns.tolist())
print("Colonne perfettamente identiche?:", dc_comics.columns.tolist() == marvel_comics.columns.tolist())


Colonne DC:     ['issue_id', 'publisher', 'series_title', 'issue_title', 'release_date', 'comic_type', 'writer', 'penciler', 'cover_artist']
Colonne Marvel: ['issue_id', 'publisher', 'series_title', 'issue_title', 'release_date', 'comic_type', 'writer', 'penciler', 'cover_artist']
Colonne perfettamente identiche?: True


In [25]:
# Salvataggio dei dataset provvisori con colonne omologate
DC_PROVVISORIO_PATH = PULITI_PATH / 'DC_Comics_provvisorio.csv'
MARVEL_PROVVISORIO_PATH = PULITI_PATH / 'Marvel_Comics_provvisorio.csv'

dc_comics.to_csv(DC_PROVVISORIO_PATH, index=False)
marvel_comics.to_csv(MARVEL_PROVVISORIO_PATH, index=False)

print(f"Dataset DC provvisorio salvato in: {DC_PROVVISORIO_PATH} ({len(dc_comics)} righe, {len(dc_comics.columns)} colonne)")
print(f"Dataset Marvel provvisorio salvato in: {MARVEL_PROVVISORIO_PATH} ({len(marvel_comics)} righe, {len(marvel_comics.columns)} colonne)")


Dataset DC provvisorio salvato in: /Users/marcogreco/ComicGraph/Dataset/Puliti/DC_Comics_provvisorio.csv (52812 righe, 9 colonne)
Dataset Marvel provvisorio salvato in: /Users/marcogreco/ComicGraph/Dataset/Puliti/Marvel_Comics_provvisorio.csv (34853 righe, 9 colonne)


In [26]:
# Lista delle colonne testuali da ripulire dagli spazi bianchi in più
text_columns = [
    'series_title',
    'issue_title',
    'release_date',
    'comic_type',
    'writer',
    'penciler',
    'cover_artist'
]

# Applicazione di str.strip() su DC e Marvel
for col in text_columns:
    dc_comics[col] = dc_comics[col].astype(str).str.strip()
    marvel_comics[col] = marvel_comics[col].astype(str).str.strip()

print("Valori unici comic_type DC:\n", dc_comics['comic_type'].unique())
print("\nValori unici comic_type Marvel (spazi rimossi):\n", marvel_comics['comic_type'].unique())


Valori unici comic_type DC:
 <StringArray>
['Category', 'Stand-Alone']
Length: 2, dtype: str

Valori unici comic_type Marvel (spazi rimossi):
 <StringArray>
[ 'Infinite Comic',           'Comic',          'Digest',         'Unknown',
   'Digital Comic', 'Trade Paperback',       'Hardcover',        'MAGAZINE',
        'Magazine',           'comic',   'DIGITAL COMIC',   'Graphic Novel']
Length: 12, dtype: str


In [27]:
righe_prima = len(marvel_comics)
# Rimuoviamo le righe che non rappresentano veri albi 
marvel_comics = marvel_comics[marvel_comics['issue_title'] != 'Unknown'].copy()
righe_eliminate = righe_prima - len(marvel_comics)
print(f"Righe fantasma eliminate da Marvel: {righe_eliminate} (atteso: 521)")
print(f"Righe Marvel rimanenti: {len(marvel_comics)}")


Righe fantasma eliminate da Marvel: 521 (atteso: 521)
Righe Marvel rimanenti: 34332


In [28]:
#DC trasformiamo in unknown i 5 valori con solo mese e i 5 valori con un numero spurio, per il restante troviamo l'anno con una regex e lasciamo solo quello
#Marvel trasformiamo i unknown tutte le date fittizzie rappresentate da valori anomali es -0001
#per entrambi andiamo a sostituire la vecchia colonna release_date con una nuova colonna release_year contenente solo l'anno di rilascio
dc_year = dc_comics['release_date'].str.extract(r'\b(19\d{2}|20\d{2})\b')[0]
marvel_year = marvel_comics['release_date'].str.extract(r'\b(19\d{2}|20\d{2})\b')[0]
# Assegniamo 'Unknown' a tutti i valori mancanti o anomali
dc_comics['release_year'] = dc_year.fillna('Unknown')
marvel_comics['release_year'] = marvel_year.fillna('Unknown')
# Rimuoviamo la vecchia colonna release_date
dc_comics = dc_comics.drop(columns=['release_date'])
marvel_comics = marvel_comics.drop(columns=['release_date'])


print("\n=== VERIFICA DC COMICS ===")
print("Anni validi:", (dc_comics['release_year'] != 'Unknown').sum())
print("Anni Unknown (66 + 5 + 5):", (dc_comics['release_year'] == 'Unknown').sum())
print("\n=== VERIFICA MARVEL COMICS ===")
print("Anni validi:", (marvel_comics['release_year'] != 'Unknown').sum())
print("Anni Unknown (543 date nulle):", (marvel_comics['release_year'] == 'Unknown').sum())
print("\nColonne identiche?:", dc_comics.columns.tolist() == marvel_comics.columns.tolist())



=== VERIFICA DC COMICS ===
Anni validi: 52736
Anni Unknown (66 + 5 + 5): 76

=== VERIFICA MARVEL COMICS ===
Anni validi: 33789
Anni Unknown (543 date nulle): 543

Colonne identiche?: True


In [29]:
import re

def clean_creators(val):
    if not isinstance(val, str) or val.strip().lower() in ('', 'unknown', 'nan'):
        return 'Unknown'
    
    # Divide per virgola, rimuove spazi all'inizio e fine e comprime spazi multipli
    names = [re.sub(r'\s+', ' ', n.strip()) for n in val.split(',')]
    
    # Rimuove elementi vuoti  es virogole in più
    names = [n for n in names if n]
    
    # Filtra via gli 'Unknown' solo se ci sono altri autori noti
    known_names = [n for n in names if n.lower() != 'unknown']
    
    # Se abbiamo trovato autori noti, uniamo quelli; altrimenti lasciamo 'Unknown'
    if known_names:
        return ', '.join(known_names)
    
    return 'Unknown'


creator_columns = ['writer', 'penciler', 'cover_artist']

for col in creator_columns:
    dc_comics[col] = dc_comics[col].apply(clean_creators)
    marvel_comics[col] = marvel_comics[col].apply(clean_creators)


print("Test rapido della logica:")
print("'Loeb, Unknown' ->", clean_creators("Loeb, Unknown"))
print("'Unknown, Jack Kirby' ->", clean_creators("Unknown, Jack Kirby"))
print("'Unknown' ->", clean_creators("Unknown"))


Test rapido della logica:
'Loeb, Unknown' -> Loeb
'Unknown, Jack Kirby' -> Jack Kirby
'Unknown' -> Unknown


In [30]:
import re
import pandas as pd

#Per unificare i due dataset dobbiamo modificare la dicitura basata sul numero di volume della DC e omologarla a quella con l'anno di partenza della marvel
#Per risolvere i 58 conflitti in cui due volumi diversi della stessa serie sono stati pubblicati nello stesso anno andiamo ad aggiungere il numero del volume a queste righe
#Marvel risolve la problematica aggiungendo inizio e fine della serie, i dati sulla fine della serie non erano presenti in maniera completa per DC e le serie attualmente in uscita sarebbero state bloccate all'ultima data relativa all'ultimo albo disponibile nel dataset
#Quindi per evitare di inserire informazioni sbagliate come la fine di una serie in corso ho attuato questa strategia ibrida per le 58 righe in collisione



#Estrazione dell'anno di pubblicazione per ogni albo DC
dc_comics['year_temp'] = pd.to_numeric(
    dc_comics['release_year'].str.extract(r'\b(19\d{2}|20\d{2})\b')[0], 
    errors='coerce'
)

#Calcolo dell'anno di debutto di ogni serie trovando l'anno minimo
series_min_year = dc_comics.groupby('series_title')['year_temp'].min()

#Funzione per separare il nome base dalla parte del Volume
def parse_series_title(title):
    # Cattura 'Vol 1', 'Volume 2' e qualsiasi specifica successiva (es. 'Vol 1 428', 'Vol 1 C-26')
    m = re.search(r'\b(Vol(?:ume)?\.?\s+.*)$', title, flags=re.IGNORECASE)
    vol_part = m.group(1).strip() if m else None
    base_part = re.sub(r'\s+Vol(?:ume)?\.?\s+.*$', '', title, flags=re.IGNORECASE).strip()
    return base_part, vol_part

#Creazione tabella per ognio serie 
series_info = pd.DataFrame({
    'original_title': series_min_year.index, 
    'min_year': series_min_year.values
})
series_info[['base_name', 'vol_str']] = [parse_series_title(t) for t in series_info['original_title']]

#Generazione del titolo nome + data inizio
series_info['standard_title'] = series_info.apply(
    lambda r: f"{r['base_name']} ({int(r['min_year'])})" if pd.notna(r['min_year']) else r['original_title'],
    axis=1
)

#Individuazione delle serie in collisione con gli anni
title_counts = series_info.groupby('standard_title')['original_title'].nunique()
colliding_titles = set(title_counts[title_counts > 1].index)

#Soluzione ibrida anno-volume per le sole collissioni
def resolve_final_title(row):
    #Manteniamo la nomenclatura originale per i 7 volumi che non hanno una data
    if pd.isna(row['min_year']):
        return row['original_title']
    
    # Se c'è collisione tra più volumi nello stesso anno, aggiungi il volume tra parentesi
    if row['standard_title'] in colliding_titles:
        vol_desc = row['vol_str'] if row['vol_str'] else 'Special'
        return f"{row['base_name']} ({int(row['min_year'])} - {vol_desc})"
    
    # Per tutte le altre usa la nuova forma con l'anno
    return row['standard_title']

series_info['final_title'] = series_info.apply(resolve_final_title, axis=1)


title_mapping = dict(zip(series_info['original_title'], series_info['final_title']))
dc_comics['series_title'] = dc_comics['series_title'].map(title_mapping)

# Rimuoviamo la colonna d'appoggio temporanea
dc_comics = dc_comics.drop(columns=['year_temp'])


print(f"Serie DC totali: {dc_comics['series_title'].nunique()} ")


Serie DC totali: 2908 


In [31]:
# 1. Verifica iniziale dei duplicati
duplicati_iniziali = dc_comics.drop(columns=['issue_id']).duplicated().sum()
print(f"Duplicati reali trovati in DC (escluso issue_id): {duplicati_iniziali}")

righe_prima = len(dc_comics)

# 2. Rimozione dei duplicati di contenuto (manteniamo solo la prima occorrenza)
content_cols = [c for c in dc_comics.columns if c != 'issue_id']
dc_comics = dc_comics.drop_duplicates(subset=content_cols, keep='first').copy()
duplicati_rimossi = righe_prima - len(dc_comics)

# 3. Rimozione degli header di sezione Fandom (anni fittizi come '1982', '1983') e dei due link editoriali spuri
is_year_header = dc_comics['issue_title'].str.match(r'^(19\d{2}|20\d{2})$', na=False)
is_spurious = dc_comics['issue_title'].isin(['Fawcett Publications', 'DC Comics'])

anomalie_rimosse = (is_year_header | is_spurious).sum()
dc_comics = dc_comics[~is_year_header & ~is_spurious].copy()

# 4. Rigenerazione degli issue_id puliti e sequenziali
dc_comics = dc_comics.reset_index(drop=True)
dc_comics['issue_id'] = 'dc_issue_' + dc_comics.index.astype(str)

# 5. Report finale di pulizia
print("\n=== REPORT PULIZIA DC COMICS ===")
print(f"Righe DC iniziali:                  {righe_prima}")
print(f"Duplicati eliminati:                {duplicati_rimossi} (atteso: 25.673)")
print(f"Intestazioni Fandom / link rimossi: {anomalie_rimosse} (atteso: 552)")
print(f"Righe DC finali valide e uniche:    {len(dc_comics)} (atteso: 26.587)")
print(f"Intervallo ID generati:             {dc_comics['issue_id'].iloc[0]} -> {dc_comics['issue_id'].iloc[-1]}")


Duplicati reali trovati in DC (escluso issue_id): 25673

=== REPORT PULIZIA DC COMICS ===
Righe DC iniziali:                  52812
Duplicati eliminati:                25673 (atteso: 25.673)
Intestazioni Fandom / link rimossi: 552 (atteso: 552)
Righe DC finali valide e uniche:    26587 (atteso: 26.587)
Intervallo ID generati:             dc_issue_0 -> dc_issue_26586


In [32]:
import os
from pathlib import Path
import pandas as pd

# 1. Cancellazione dei file provvisori dal disco
DC_PROVVISORIO_PATH = PULITI_PATH / 'DC_Comics_provvisorio.csv'
MARVEL_PROVVISORIO_PATH = PULITI_PATH / 'Marvel_Comics_provvisorio.csv'

for provv_file in [DC_PROVVISORIO_PATH, MARVEL_PROVVISORIO_PATH]:
    if provv_file.exists():
        provv_file.unlink()
        print(f"File provvisorio rimosso: {provv_file.name}")

# 2. Definizione dell'ordine standard finale delle 9 colonne
FINAL_COLUMNS_ORDER = [
    'issue_id',
    'publisher',
    'series_title',
    'issue_title',
    'release_year',
    'comic_type',
    'writer',
    'penciler',
    'cover_artist'
]

# Allineamento dell'ordine delle colonne prima dell'unione
dc_comics = dc_comics[FINAL_COLUMNS_ORDER]
marvel_comics = marvel_comics[FINAL_COLUMNS_ORDER]

# 3. Concatenazione dei due dataset
fumetti_clean = pd.concat([dc_comics, marvel_comics], ignore_index=True)

# 4. Salvataggio del dataset definitivo unificato
FUMETTI_CLEAN_PATH = PULITI_PATH / 'Fumetti_clean.csv'
fumetti_clean.to_csv(FUMETTI_CLEAN_PATH, index=False)

# 5. Stampa riassuntiva di verifica
print("\n" + "="*45)
print("  DATASET FUMETTI UNIFICATO CREATO CON SUCCESSO!")
print("="*45)
print(f"Percorso file salvato:     {FUMETTI_CLEAN_PATH}")
print(f"Dimensioni totali:         {fumetti_clean.shape[0]} righe x {fumetti_clean.shape[1]} colonne")
print("\nDistribuzione per Editore:")
print(fumetti_clean['publisher'].value_counts().to_string())
print(f"\nDuplicati di 'issue_id':   {fumetti_clean['issue_id'].duplicated().sum()} (0 attesi)")
print(f"Colonne finali:            {fumetti_clean.columns.tolist()}")
print("="*45)


File provvisorio rimosso: DC_Comics_provvisorio.csv
File provvisorio rimosso: Marvel_Comics_provvisorio.csv

  DATASET FUMETTI UNIFICATO CREATO CON SUCCESSO!
Percorso file salvato:     /Users/marcogreco/ComicGraph/Dataset/Puliti/Fumetti_clean.csv
Dimensioni totali:         60919 righe x 9 colonne

Distribuzione per Editore:
publisher
Marvel    34332
DC        26587

Duplicati di 'issue_id':   0 (0 attesi)
Colonne finali:            ['issue_id', 'publisher', 'series_title', 'issue_title', 'release_year', 'comic_type', 'writer', 'penciler', 'cover_artist']
